# Piloto de fine-tuning — acortar las pausas

Verificación rápida: coge audios ya generados en Drive (por defecto, las frases de prueba del LoRA de Qwen) y
**acorta solo los silencios**, sin tocar el habla:

- el silencio del principio queda en 0,05 s y el del final en 0,15 s;
- cada pausa interna de más de `PAUSA_MAX` se queda en `PAUSA_MAX` (se le quita el centro);
- las pausas más cortas (comas, respiraciones) no se tocan.

Los escuchas aquí mismo, uno debajo de otro, con su duración y su velocidad en caracteres por segundo. Como
referencia, Naval en inglés habla a unos 19,5 c/s y Qwen en español sale a unos 15. Lo que más se nota es el
bloque de 9 frases (n25): ahí se acumulan las pausas entre frases.

No genera nada con el modelo ni guarda nada en Drive, salvo que pongas `GUARDAR = True`.

**Entorno: CPU, sin GPU** (*Entorno de ejecución → Cambiar tipo de entorno → CPU*). Tarda un minuto.
Ejecuta las celdas en orden; la última desconecta el entorno.

In [ ]:
# 1. Parámetros
CARPETA = "piloto/lora_qwen/B/prueba"     # dentro de fine_tunned_naval
VERSIONES = ["paso_0158"]                 # añade "base" para comparar con el modelo sin LoRA
FRASES = ["n12", "n18", "n25"]
PAUSAS_MAX = [0.25, 0.18, 0.12]           # segundos; 0,12 es el valor fijado para la fábrica
GUARDAR = False                           # True: deja las versiones en <CARPETA>/pausas/ en Drive

In [ ]:
# 2. Preparar: código del repo, Drive y los textos de las frases
import os, sys, subprocess
from pathlib import Path
RAMA = "claude/quirky-dirac-x18c3n"
if os.path.isdir("/content/repo"):
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, "https://github.com/DreiSup/spanishnav.al-podcast", "/content/repo"], check=True)
sys.path[:0] = ["/content/repo/criba", "/content/repo/piloto"]
import comun, ritmo

comun.montar_drive()
BASE = comun.DRIVE / CARPETA
items, _ = comun.cargar_items()
TEXTO = {it["id"]: " ".join(it["frases"]) for it in items}
faltan = [str(BASE / v / f"{f}.wav") for v in VERSIONES for f in FRASES if not (BASE / v / f"{f}.wav").is_file()]
if faltan:
    raise SystemExit(f"Faltan audios en Drive, por ejemplo {faltan[0]}")
print("RESULTADO: OK")

In [ ]:
# 3. Acortar las pausas y escuchar
import soundfile as sf
from IPython.display import Audio, Markdown, display

for f in FRASES:
    display(Markdown(f"### {f}\n*{TEXTO[f]}*"))
    for v in VERSIONES:
        audio, sr = sf.read(str(BASE / v / f"{f}.wav"), dtype="float32")
        for pm in [None] + PAUSAS_MAX:
            corto = audio if pm is None else ritmo.acortar_pausas(audio, sr, pausa_max=pm)
            seg = len(corto) / sr
            nombre = "original" if pm is None else f"pausas ≤ {pm:.2f} s"
            ahorro = "" if pm is None else f", {len(audio) / sr - seg:.1f} s menos"
            display(Markdown(f"**{v} · {nombre}** — {seg:.1f} s, {len(TEXTO[f]) / seg:.1f} c/s{ahorro}"))
            display(Audio(corto, rate=sr))
            if GUARDAR and pm is not None:
                destino = BASE / "pausas" / v
                destino.mkdir(parents=True, exist_ok=True)
                sf.write(str(destino / f"{f}_p{pm:.2f}.wav"), corto, sr, subtype="PCM_16")

In [ ]:
# 4. Desconectar (sube a Drive lo pendiente si has guardado algo)
comun.desconectar()